[⬅ Back to the examples](https://anywidgetinstruments.github.io/anywidget-instruments-industrial/examples/)

# Tank level supervision with alarms

A tank is filled by a pump and drained through a valve. The level has
**LO / HI / HIHI** alarm limits (with deadband) that feed an alarm banner
following ISA-18.2 (acknowledge them with **ACK**). In AUTO, a hysteresis
controller starts the pump below 30 % and stops it above 70 %. Click the
pump or the valve to open its faceplate and operate it in MANUAL.

> **The code is hidden.** To see a cell's code, click the `⋯` bar above its result; to see all of it, choose **View ▸ Expand All Code**.

**References:** ISA-18.2 / IEC 62682 (alarm banner), ISA-5.1 (tags, process
symbols). These standards inspired the widgets; the library does not claim
conformity with them ([Standards and
references](https://anywidgetinstruments.github.io/anywidget-instruments-industrial/standards/)).

In [ ]:
import os
import threading
import time

import anywidget_instruments as ai

# How long the live simulation runs (seconds). The documentation build and the
# test-suite set AWI_EXAMPLE_SECONDS to run the examples briefly.
RUN_SECONDS = float(os.environ.get("AWI_EXAMPLE_SECONDS", "600"))

In [ ]:
ai.theme_switch()  # light / system / dark theme (STYLE-008)

In [ ]:
tank = ai.Tank(
    40,
    min=0,
    max=100,
    unit="%",
    label="TK-101 level",
    lo=15,
    hi=85,
    hihi=95,
    deadband=2,
    show_limits=True,
    markers=[30, 70],
    size=(120, 220),
)
pump = ai.Pump(tag="P-101", label="P-101", simulate=True)
valve = ai.Valve(tag="XV-102", label="XV-102", value="open", position=60, simulate=True)
demand = ai.Knob(60, min=0, max=100, step=1, unit="%", label="Outflow valve %", size=(120, 120))
level_hi = ai.LED(label="HI", on_color="#f59e0b", shape="square", size=(36, 36))
level_hihi = ai.LED(label="HIHI", on_color="#dc2626", shape="square", blink=True, size=(36, 36))
banner = ai.AlarmBanner(label="Alarms", size=(560, 150))
trend = ai.WaveformChart(
    n_traces=1,
    history=600,
    dt=0.2,
    x_unit="s",
    unit="%",
    y_min=0,
    y_max=100,
    label="Level trend",
    size=(560, 180),
    traces=[{"name": "level"}],
)
syn = ai.SynopticCanvas(size=(560, 300), label="Synoptic")
syn.add(pump, x=40, y=150)
syn.add(tank, x=220, y=5)
syn.add(valve, x=420, y=170)
syn.add(ai.Panel([level_hi, level_hihi], columns=2), x=360, y=20)
# pipe runs (canvas coordinates): pump outlet -> tank top, tank bottom -> valve -> drain
inflow = syn.add_pipe([(123, 196), (190, 196), (190, 60), (272, 60)], flow=False)
outflow = syn.add_pipe([(314, 225), (380, 225), (380, 241), (437, 241)], flow=True)
syn.add_pipe([(501, 241), (555, 241)], flow=True)

ai.Panel([syn, ai.Panel([demand, banner], columns=1), trend], columns=2)

In [ ]:
ALARM_TEXT = {"lo": "Level low", "hi": "Level high", "hihi": "Level very high"}


@tank.on_change(names="alarm_level")
def _alarms(change):
    """Raise / clear banner alarms from the tank alarm level (ALARM-004)."""
    new, old = change["new"], change["old"]
    if old in ALARM_TEXT:
        banner.clear_alarm(f"TK-101.{old.upper()}")
    if new in ALARM_TEXT:
        priority = "critical" if new == "hihi" else "high"
        banner.raise_alarm(
            f"TK-101.{new.upper()}", ALARM_TEXT[new], source="TK-101", priority=priority
        )
    level_hi.value = new in ("hi", "hihi")
    level_hihi.value = new == "hihi"


@banner.on_acknowledge
def _log_ack(event):
    print("acknowledged:", event["alarm_id"])


@demand.on_change
def _valve_position(change):
    valve.position = change["new"]
    valve.value = "open" if change["new"] > 0 else "closed"

In [ ]:
def simulate(duration=RUN_SECONDS, dt=0.2):
    t_end = time.time() + duration
    while time.time() < t_end:
        if pump.auto:  # hysteresis controller
            if tank.value < 30 and pump.value == "stopped":
                pump.value = "running"
            elif tank.value > 70 and pump.value == "running":
                pump.value = "stopped"
        q_in = 6.0 if pump.value == "running" else 0.0
        q_out = 4.0 * (valve.position or 0) / 100 if valve.value == "open" else 0.0
        with ai.batch():
            tank.value = min(100.0, max(0.0, tank.value + (q_in - q_out) * dt))
            syn.set_flow(inflow, q_in > 0)
            syn.set_flow(outflow, q_out > 0)
            trend.append(tank.value)
        time.sleep(dt)


threading.Thread(target=simulate, daemon=True).start()

**Things to try**

* Put the pump in **MANUAL** from its faceplate and start it: the level
  rises through HI and HIHI; watch the banner and acknowledge the alarms.
* Close the outflow valve (knob to 0 %) and observe the trend.
* The alarm colours and blink patterns are documented in
  `docs/alarm-conventions.md`.